In [1]:
print('hello world')

hello world


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from joblib import Parallel, delayed
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
import scipy.stats as stats
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder,PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [3]:
df_2022=pd.read_csv('fifa_ranking_2022-10-06.csv')
df_2025=pd.read_csv('fifa_ranking_2026-06-08.csv')
df_matches=pd.read_csv('matches_1930_2022.csv')
df_shedule=pd.read_csv('schedule_2026.csv')
df_world_cup=pd.read_csv('world_cup.csv')



In [4]:
#trying to find total teams from 1930 to 2022
match_teams = set(df_matches['home_team'].dropna().unique()).union(set(df_matches['away_team'].dropna().unique()))

# total teams in 2022 
ranking_teams = set(df_2022['team'].dropna().unique())

# team that are not in 2022 bit were there in btw 1930 to 2022
missing_in_rankings = sorted(list(match_teams - ranking_teams))

print(f"Total unique teams in Matches: {len(match_teams)}")
print(f"Total unique teams in Rankings: {len(ranking_teams)}")
print(f"Number of mismatches: {len(missing_in_rankings)}")
print("Mismatched Teams:", missing_in_rankings)

Total unique teams in Matches: 86
Total unique teams in Rankings: 211
Number of mismatches: 11
Mismatched Teams: ['Czech Republic', 'Czechoslovakia', 'Dutch East Indies', 'FR Yugoslavia', 'Germany DR', 'Serbia and Montenegro', 'Soviet Union', 'United States', 'West Germany', 'Yugoslavia', 'Zaire']


In [5]:
missing_in_rankings



['Czech Republic',
 'Czechoslovakia',
 'Dutch East Indies',
 'FR Yugoslavia',
 'Germany DR',
 'Serbia and Montenegro',
 'Soviet Union',
 'United States',
 'West Germany',
 'Yugoslavia',
 'Zaire']

In [8]:


country_mapping = {
    'United States': 'USA',
    'Czech Republic': 'Czechia',
    'Czechoslovakia': 'Czechia',
    'Dutch East Indies': 'Indonesia',
    'FR Yugoslavia': 'Serbia',
    'Germany DR': 'Germany',
    'Serbia and Montenegro': 'Serbia',
    'Soviet Union': 'Russia',
    'West Germany': 'Germany',
    'Yugoslavia': 'Serbia',
    'Zaire': 'Congo DR'
}

df_matches['home_team'] = df_matches['home_team'].replace(country_mapping)
df_matches['away_team'] = df_matches['away_team'].replace(country_mapping)

In [9]:
print(df_matches['home_penalty'].value_counts(dropna=False))
print(df_matches['away_penalty'].value_counts(dropna=False))

home_penalty
NaN    929
3.0     13
4.0     11
5.0      5
1.0      3
2.0      2
0.0      1
Name: count, dtype: int64
away_penalty
NaN    929
3.0     11
4.0     10
2.0      9
5.0      3
0.0      1
1.0      1
Name: count, dtype: int64


 ## Rounds where a draw is allowed
draw_rounds = [
   'Group stage',
    'First group stage',
    'Second group stage']
#
def get_match_result(row):
    # Normal time / Extra time result
    if row['home_score'] > row['away_score']:
          return 'Home Win'
    elif row['home_score'] < row['away_score']:
       return 'Away Win'
    # Scores are equal
    else:
        # Draw is allowed in these rounds
        if row['Round'] in draw_rounds:
            return 'Draw'
        # Knockout rounds -> decide by penalties
        else:
            if pd.notna(row['home_penalty']) and pd.notna(row['away_penalty']):
                if row['home_penalty'] > row['away_penalty']:
                    return 'Home Win'
                elif row['home_penalty'] < row['away_penalty']:
                    return 'Away Win'
            #If penalty data is unavailable
            return 'Draw'


 #Create the new column
df_matches['Match_Result'] = df_matches.apply(get_match_result, axis=1)

 #Check the result distribution
print(df_matches['Match_Result'].value_counts())

"While technically precise to implement the above code—given that match outcomes vary between round-robin stages (which allow draws) and knockout phases (which do not)—the simpler logic below will be utilized for this model.

In [10]:
df_matches['Match_Result'] = np.where(
    df_matches['home_score'] > df_matches['away_score'], 0,
    np.where(
        df_matches['home_score'] < df_matches['away_score'], 1,
        2
    )
)
print(df_matches['Match_Result'].value_counts())

Match_Result
0    532
1    218
2    214
Name: count, dtype: int64


In [11]:

train_data = df_matches[df_matches['Year'] <= 2018].copy()
test_data = df_matches[df_matches['Year'] == 2022].copy()

Since we cannot use home_score and away_score in the model as practically we dont know the scores before the match it will be a data leakage so we will use a new column that is the rolling average of goals scored up to that point in time

In [12]:
train_data['Date'] = pd.to_datetime(train_data['Date'])
test_data['Date'] = pd.to_datetime(test_data['Date'])

In [13]:
home_goals = train_data[['home_team', 'home_score']].rename(
    columns={'home_team': 'team', 'home_score': 'goals_scored'}
)

away_goals = train_data[['away_team', 'away_score']].rename(
    columns={'away_team': 'team', 'away_score': 'goals_scored'}
)

all_goals = pd.concat([home_goals, away_goals], ignore_index=True)

team_avg_goals = all_goals.groupby('team')['goals_scored'].mean()

train_data['home_avg_goals'] = train_data['home_team'].map(team_avg_goals)
train_data['away_avg_goals'] = train_data['away_team'].map(team_avg_goals)
test_data['home_avg_goals'] = test_data['home_team'].map(team_avg_goals)
test_data['away_avg_goals'] = test_data['away_team'].map(team_avg_goals)

overall_avg = all_goals['goals_scored'].mean()

train_data[['home_avg_goals', 'away_avg_goals']] = (
    train_data[['home_avg_goals', 'away_avg_goals']].fillna(overall_avg)
)

test_data[['home_avg_goals', 'away_avg_goals']] = (
    test_data[['home_avg_goals', 'away_avg_goals']].fillna(overall_avg)
)


Iintroducing a new feature to capture each team's historical World Cup experience. In international football, experienced teams typically possess a higher probability of winning because they have adapted to tournament pressure and learned from past mistakes—an advantage that debutant or less-experienced teams do not share.

In [ ]:
import pandas as pd

home_years = train_data[['Year', 'home_team']].rename(columns={'home_team': 'team'})
away_years = train_data[['Year', 'away_team']].rename(columns={'away_team': 'team'})


all_appearances = pd.concat([home_years, away_years]).drop_duplicates()


world_cup_experience = all_appearances.groupby('team')['Year'].count()


train_data['home_wc_experience'] = train_data['home_team'].map(world_cup_experience)
train_data['away_wc_experience'] = train_data['away_team'].map(world_cup_experience)


test_data['home_wc_experience'] = test_data['home_team'].map(world_cup_experience)
test_data['away_wc_experience'] = test_data['away_team'].map(world_cup_experience)


train_data[['home_wc_experience', 'away_wc_experience']] = (
    train_data[['home_wc_experience', 'away_wc_experience']].fillna(0)
)
test_data[['home_wc_experience', 'away_wc_experience']] = (
    test_data[['home_wc_experience', 'away_wc_experience']].fillna(0)
)

"We are factoring in a Home Field Advantage weight into our model. Teams playing in their host country benefit from both familiarity with local pitches and significant crowd support. While stadium attendance consists of opposing fans as well, we assume that total attendance scales proportionally with home-team support, amplifying their advantage as match attendance increases."

In [15]:

train_data['home_is_host'] = (train_data['home_team'] == train_data['Host']).astype(int)
train_data['away_is_host'] = (train_data['away_team'] == train_data['Host']).astype(int)

test_data['home_is_host'] = (test_data['home_team'] == test_data['Host']).astype(int)
test_data['away_is_host'] = (test_data['away_team'] == test_data['Host']).astype(int)

train_data['home_adrenaline_boost'] = train_data['home_is_host'] 
train_data['away_adrenaline_boost'] = train_data['away_is_host'] 

test_data['home_adrenaline_boost'] = test_data['home_is_host'] 
test_data['away_adrenaline_boost'] = test_data['away_is_host'] 

train_data[['home_adrenaline_boost', 'away_adrenaline_boost']] = (
    train_data[['home_adrenaline_boost', 'away_adrenaline_boost']].fillna(0)
)

test_data[['home_adrenaline_boost', 'away_adrenaline_boost']] = (
    test_data[['home_adrenaline_boost', 'away_adrenaline_boost']].fillna(0)
)


"To avoid data leakage, we cannot directly include match-specific penalty outcomes in our feature set. Instead, we capture a team's proficiency by engineering a feature for total historical penalties played. Teams with higher penalty experience are statistically better prepared to handle high-pressure shootout scenarios, giving them a distinct advantage if a match transitions to penalties."

In [ ]:
import pandas as pd

home_penalty_exp = (
    train_data[train_data['home_penalty'].notna()]
    .groupby('home_team')
    .size()
)

away_penalty_exp = (
    train_data[train_data['away_penalty'].notna()]
    .groupby('away_team')
    .size()
)


team_penalty_exp = home_penalty_exp.add(
    away_penalty_exp,
    fill_value=0
)


train_data['home_penalty_exp'] = (
    train_data['home_team']
    .map(team_penalty_exp)
    .fillna(0)
)

train_data['away_penalty_exp'] = (
    train_data['away_team']
    .map(team_penalty_exp)
    .fillna(0)
)


test_data['home_penalty_exp'] = (
    test_data['home_team']
    .map(team_penalty_exp)
    .fillna(0)
)

test_data['away_penalty_exp'] = (
    test_data['away_team']
    .map(team_penalty_exp)
    .fillna(0)
)

In [17]:
train_data['home_yellow_red_card'].dropna().sample(5).tolist()

['Cyril Domoraud · 90+2',
 'Arthur Numan · 76',
 'Basil Gorgis · 52',
 'Avery John · 46',
 'Jan Polák · 45+2']

In [18]:
train_data['home_red_card'].sample(10)

926    NaN
643    NaN
206    NaN
326    NaN
403    NaN
159    NaN
476    NaN
848    NaN
247    NaN
887    NaN
Name: home_red_card, dtype: str

In [ ]:
import pandas as pd


def count_cards_vectorized(series, delimiter=';'):
    
    return series.fillna('').astype(str).apply(lambda x: len([i for i in x.split(delimiter) if i.strip()]) if x else 0)


card_cols = {
    'home_yellow_cards': ('home_yellow_card_long', ';'),
    'away_yellow_cards': ('away_yellow_card_long', ';'),
    'home_red_cards': ('home_red_card', '.'),          # Double check if '.' is correct!
    'away_red_cards': ('away_red_card', '.'),
    'home_yellow_red_cards': ('home_yellow_red_card', '.'),
    'away_yellow_red_cards': ('away_yellow_red_card', '.')
}

for new_col, (raw_col, sep) in card_cols.items():
    train_data[new_col] = count_cards_vectorized(train_data[raw_col], sep)


for side in ['home', 'away']:
    train_data[f'{side}_match_card_score'] = (
        train_data[f'{side}_yellow_cards']
        + 2 * train_data[f'{side}_red_cards']
        + 2 * train_data[f'{side}_yellow_red_cards']
    )


home_cards = train_data[['home_team', 'home_match_card_score']].rename(columns={'home_team': 'team', 'home_match_card_score': 'card_score'})
away_cards = train_data[['away_team', 'away_match_card_score']].rename(columns={'away_team': 'team', 'away_match_card_score': 'card_score'})

team_discipline_score = pd.concat([home_cards, away_cards], ignore_index=True).groupby('team')['card_score'].mean()
global_mean = team_discipline_score.mean()

for side in ['home', 'away']:
    train_data[f'{side}_discipline_score'] = train_data[f'{side}_team'].map(team_discipline_score)
    test_data[f'{side}_discipline_score'] = test_data[f'{side}_team'].map(team_discipline_score).fillna(global_mean)

In [ ]:

all_teams = pd.concat([
    train_data['home_team'],
    train_data['away_team']
]).unique()


team_to_id = {team: idx for idx, team in enumerate(sorted(all_teams))}


train_data['home_team_id'] = train_data['home_team'].map(team_to_id)
train_data['away_team_id'] = train_data['away_team'].map(team_to_id)

test_data['home_team_id'] = test_data['home_team'].map(team_to_id).fillna(-1).astype(int)
test_data['away_team_id'] = test_data['away_team'].map(team_to_id).fillna(-1).astype(int)

In [ ]:
train_data['Venue_City'] = train_data['Venue'].str.split(',').str[-1].str.strip()
test_data['Venue_City'] = test_data['Venue'].str.split(',').str[-1].str.strip()

train_data['Referee'] = train_data['Referee'].fillna('Unknown')

test_data['Referee'] = test_data['Referee'].fillna('Unknown')

In [22]:
test_data.columns

Index(['home_team', 'away_team', 'home_score', 'home_xg', 'home_penalty',
       'away_score', 'away_xg', 'away_penalty', 'home_manager', 'home_captain',
       'away_manager', 'away_captain', 'Attendance', 'Venue', 'Officials',
       'Round', 'Date', 'Score', 'Referee', 'Notes', 'Host', 'Year',
       'home_goal', 'away_goal', 'home_goal_long', 'away_goal_long',
       'home_own_goal', 'away_own_goal', 'home_penalty_goal',
       'away_penalty_goal', 'home_penalty_miss_long', 'away_penalty_miss_long',
       'home_penalty_shootout_goal_long', 'away_penalty_shootout_goal_long',
       'home_penalty_shootout_miss_long', 'away_penalty_shootout_miss_long',
       'home_red_card', 'away_red_card', 'home_yellow_red_card',
       'away_yellow_red_card', 'home_yellow_card_long',
       'away_yellow_card_long', 'home_substitute_in_long',
       'away_substitute_in_long', 'Match_Result', 'home_avg_goals',
       'away_avg_goals', 'home_wc_experience', 'away_wc_experience',
       'home_is_host

In [23]:
used_cols=['home_xg','away_xg','home_team_id','home_manager', 'away_manager','away_team_id',
           'Match_Result', 'home_avg_goals','Attendance', 'Venue_City','Round', 
       'away_avg_goals', 'home_wc_experience', 'away_wc_experience',
       'home_is_host', 'away_is_host', 'home_adrenaline_boost','Referee',
       'away_adrenaline_boost', 'home_penalty_exp', 'away_penalty_exp', 'Host', 'Year',
       'home_discipline_score', 'away_discipline_score']

In [24]:

X_train = train_data[used_cols]
y_train = train_data['Match_Result']

# Create Testing Inputs and Targets (2022 World Cup)
X_test = test_data[used_cols]
y_test = test_data['Match_Result']



#FROM HERE REGRESSION MODEL IS BEING MADE 

In [ ]:
test_data['Total_goals']=test_data['home_score']+test_data['away_score']
train_data['Total_goals']=train_data['home_score']+train_data['away_score']

In [ ]:
used_cols=['home_team_id','home_manager', 'away_manager','away_team_id',
           'Total_goals', 'home_avg_goals','Attendance', 'Venue_City','Round', 
       'away_avg_goals', 'home_wc_experience', 'away_wc_experience',
       'home_is_host', 'away_is_host', 'home_adrenaline_boost','Referee',
       'away_adrenaline_boost', 'home_penalty_exp', 'away_penalty_exp', 'Host', 'Year',
       'home_discipline_score', 'away_discipline_score']
X_train = train_data[used_cols]
y_train = train_data['Total_goals']

# Create Testing Inputs and Targets (2022 World Cup)
X_test = test_data[used_cols]
y_test = test_data['Total_goals']


In [26]:
y_test


0      6
1      4
2      4
3      6
4      2
      ..
59     8
60    12
61     0
62     2
63     0
Name: Total_goals, Length: 64, dtype: int64

In [28]:
label_cols=['away_manager','Venue_City','home_manager','Referee']


to_stande=['home_xg','away_xg', 'home_avg_goals','Attendance', 
       'away_avg_goals', 'home_wc_experience', 'away_wc_experience',
       'home_is_host', 'away_is_host', 'home_adrenaline_boost','home_team_id','away_team_id',
       'away_adrenaline_boost', 'home_penalty_exp', 'away_penalty_exp', 'Year',
       'home_discipline_score', 'away_discipline_score']

onehot_cols = [
    'Round',
    'Host'
]


# ----------------------------
# Column Transformer
# ----------------------------

preprocessor = ColumnTransformer(
    transformers=[

        (
            'ordinal',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            label_cols
        ),

        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore'
            ),
            onehot_cols
        ),

        (
            'numeric',
            StandardScaler(),
            to_stande
        )

    ],
    remainder='drop'
)

# ----------------------------
# Complete Pipeline
# -------------------------
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import numpy as np

# Random Forest Regressor
rf = RandomForestRegressor(random_state=42)

# Corrected Parameter Grid for Regression
param_grid = {
    'n_estimators': [100, 200, 300],
    
    # Regression criteria (how it measures splits)
    'criterion': ['squared_error', 'absolute_error'], 
    
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    
    # Added 1.0 (uses all features), which is often the best for regression
    'max_features': ['sqrt', 'log2', 1.0], 
    
    'bootstrap': [True, False]
    
    # NOTE: class_weight has been completely removed!
}

# Grid Search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    
    # Regression scoring (Scikit-learn uses negative MSE for optimization)
    scoring='neg_mean_squared_error', 
    
    n_jobs=-1,
    verbose=2
)

# Train (Make sure your y_train is the continuous Total_Goals column!)
grid_search.fit(X_train_processed, y_train)

# Best Model
best_rf = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

# Convert the negative MSE score back into a readable positive RMSE
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"\nBest CV RMSE: {best_rmse:.2f} goals")

# Prediction
y_pred = best_rf.predict(X_test_processed)

print(accuracy(y_test,y_pred))

Fitting 5 folds for each of 1296 candidates, totalling 6480 fits


In [ ]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)
print(accuracy(y_test,y_pred))
print(f"MAE: {mae:.4f}")
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")

MAE: 2.2854
RMSE: 2.9387
